# Topic Modeling
### Bert Sentence Embedding, LDA, and K-means Clustering

#### Concatenate the sentence embeddings with LDA, use an autoencoder to get the latent representation and perform K-means to perform clustering on the data

#### After the sentences have been assigned to their clusters, we can extract the candidate topics for each cluster using tokens of it's sentences

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')


In [ ]:
% cd /content/drive/My Drive/Thesis_colab

In [ ]:
! pip install transformers
! pip install pandas
! pip install torch
! pip install nltk
! pip install gensim
! pip install matplotlib
! pip install umap-learn
! pip install wordcloud
! pip install sklearn
! pip install tqdm
! pip install stop_words
! pip install xlrd
! pip install langdetect
! pip install openpyxl

In [2]:
import nltk
import numpy as np
import statistics
from sklearn.decomposition import PCA
from collections import Counter

In [ ]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

### 1. Reading and preprocessing the data

In [ ]:
from utilities.document_preprocessing import preprocess_documents

#### The preprocess_document function reads the sentences from the file and generates tokens from each sentences.

In [3]:
Verbose = False

In [ ]:
# Read new data file
file = "data/PMP_Skills_Matching_Pilot_2018_2019_small.xlsx"

ids, sentences, token_lists = preprocess_documents(file, verbose = Verbose)
num_docs = len(ids)

In [4]:
#read the previously processed data
import pandas as pd
data = pd.read_pickle("output/preprocessed_data/use_this_data.pkl")
sentences = list(data['sentences'])
token_lists = list(data['token_list'])
ids = list(data['id'])
num_docs = len(ids)

In [6]:
len(ids)

205

## 2. Topic Modeling

In [7]:
from models.topic_model import Topic_Model

/home/anum/anaconda3/envs/nlp/lib/python3.8/site-packages/umap/spectral.py:4: NumbaDeprecationWarning: No direct replacement for 'numba.targets' available. Visit https://gitter.im/numba/numba-dev to request help. Thanks!
  import numba.targets


#### The topic_Model class does the follow
- Extract Bert sentence embeddings for all sentences (Local features)
- get topic distribution for each sentence using LDA
- concatenate both and learn the latent representation using an auto-encoder
- cluster the latent representation into K clusters

In [11]:
# define the numbe of clusters
k = 2

In [12]:
# Define the topic model object
TopicModelObject = Topic_Model(k = k, verbose = Verbose, embedding = None, 
                               saved_embedding_dir = "output/preprocessed_data/bert_embeddings.json", embedding_batch_size = 100)

In [ ]:
# for Colab
# Define the topic model object
TopicModelObject = Topic_Model(k = k, verbose = Verbose, embedding = 'bert', 
                               saved_embedding_dir = "output/embeddings/2020_06_29_20_24_15_bert/bert_embeddings.json", embedding_batch_size = 100)

In [13]:
# Fit the topic model
TopicModelObject.fit(sentences, token_lists)

Getting vector representations for LDA 
Getting vector representations for LDA. Done! 

 LDA Done.


## 3. Analysis on learned features

#### Preprocessing (Dimensionality Reduction) for visualization

Perform Dimensionality reduction on the learned features using the following steps:

1) Merge all the vectors of each employee into one by taking average

2) Assign the class label to each employee based on argmax on label counts.

3) Further compress the average feature vector into 2d/3d space for visualization.

In [37]:
# Getting feature vectors and cluster labels assigned by the model
if TopicModelObject.method == 'LDA':
    feature_vector = TopicModelObject.vec[TopicModelObject.method]
    # for each sentence, find out the cluster it belongs to
    cluster_label_per_document = np.argmax(np.array(feature_vector), axis=1)
else:
    cluster_label_per_document = np.array(TopicModelObject.cluster_model.labels_)
    feature_vector = TopicModelObject.vec[TopicModelObject.method]
    feature_vector = feature_vector.cpu().numpy()

unique_ids = list(set(ids))
num_employee = len(unique_ids)

In [38]:
# Create place holder for the analysis
averaged_employee_features = pd.DataFrame(None, index=pd.Index(unique_ids, name='Employee id'),
                                          columns=pd.Index(
                                              ['feature_vector', 'cluster', 'feature_vector_2d', 'feature_vector_3d'],
                                              name='Compressed_features'))

In [39]:
# get the indices of all objectives/documents/sentence for each employee (for later use)
employee_objective_indices = dict()
for employee in unique_ids:
    employee_indices = [i for i, e in enumerate(ids) if (ids[i] == employee) ]
    employee_objective_indices[employee] = employee_indices

In [40]:
# 1 & 2 take average of all feature vectors belonging to an employee and assigning cluster label by taking the mode

for employee in unique_ids:
    #get all feature vectors belonging to an employee
    feature_vector_per_employee = feature_vector[employee_objective_indices[employee]]
    # get label for all feature vectors belonging to that employee
    clusters_per_employee = cluster_label_per_document[employee_objective_indices[employee]]  
    
    #take mean of all feature vectors
    averaged_employee_features['feature_vector'][employee] = np.mean(feature_vector_per_employee, axis=0)
    
    # pick the label with most occurance for that employee
    c = Counter(clusters_per_employee)
    averaged_employee_features['cluster'][employee] = c.most_common(1)[0][0]
    
averaged_employee_features

Compressed_features,feature_vector,cluster,feature_vector_2d,feature_vector_3d
Employee id,,,,
*100001,"[0.34402211212242645, 0.6559778780986866]",1,NaN,NaN
*100002,"[0.9503570113863263, 0.04964299127459526]",0,NaN,NaN
*100000,"[0.37800274662857436, 0.6219972517847279]",1,NaN,NaN


In [41]:
# 3) doing dimensionality reduction into 2d/3d Space
from sklearn.manifold import TSNE

#todo: find a more permanent solution for this
# panda returns a array of array for the column feature vector. Converting from array of array to 2d array
feature_vector_ = np.stack(averaged_employee_features['feature_vector'].to_numpy())

# reduce dimension of feature vectors into 2
feature_vector_2d = TSNE(n_components=2).fit_transform(feature_vector_)
averaged_employee_features['feature_vector_2d'] = list(feature_vector_2d)

# reduce dimension of feature vectors into 3
feature_vector_3d = TSNE(n_components=3).fit_transform(feature_vector_)
averaged_employee_features['feature_vector_3d'] = list(feature_vector_3d)


averaged_employee_features

Compressed_features,feature_vector,cluster,feature_vector_2d,feature_vector_3d
Employee id,,,,
*100001,"[0.34402211212242645, 0.6559778780986866]",1,"[-1180.7112, -1455.3623]","[395.57404, -366.78055, -456.86795]"
*100002,"[0.9503570113863263, 0.04964299127459526]",0,"[3887.304, -775.20966]","[-730.16144, 485.2808, -341.0601]"
*100000,"[0.37800274662857436, 0.6219972517847279]",1,"[961.1868, 1755.8652]","[334.4742, -118.39786, 797.86957]"


## 4.1 Visualization: Word Cloud

In [ ]:
from utilities.visualizations import get_wordcloud
from utilities.utils import generate_n_gram
%matplotlib inline

In [ ]:
#Getting indexes of sentences belonging to each cluster
sentence_label_dict = dict()

if TopicModelObject.method == 'LDA':
    labels = np.argmax(TopicModelObject.vec['LDA'], axis=1)
else:
    labels = TopicModelObject.cluster_model.labels_

for label in set(labels):
    sentence_label_dict[label] = [idx for idx, l in enumerate(labels) if l == label]

In [ ]:
### Top words from LDA
if TopicModelObject.method == 'LDA':
    
    lda_class_distribution = TopicModelObject.vec["LDA"]
    labels = np.argmax(lda_class_distribution, axis=1)
    

    # To get the top words for each topic, this method only uses LDA, doesn't utilize Bert or K mean clustering
    TopicModelObject.get_top_words_lda(top=10)

    for k , cluster_word_distribution in TopicModelObject.get_top_words_lda(top=100).items():
        get_wordcloud(TopicModelObject, word_distribution = cluster_word_distribution, topic = k )
# Get top words from autoencoder (LDA + Embeddings) and create word clouds on n grams
else:
    grams = 2
    for ng in range(grams):
        print ("WordClouds for {} grams".format(str(ng+1)))
        for i in range(TopicModelObject.k):

            k_sentences = np.array(sentences)[sentence_label_dict[i]]
            ngram_token_with_frequency = generate_n_gram(k_sentences, n=ng+1, separator = '-' ,
                                        keep_stopwords= False, stem=False, keep_nouns_only=False)
    
            get_wordcloud(TopicModelObject, topic=i, word_count_dict = ngram_token_with_frequency)
    

## 4.2 Visualization: Scatter plot

Visualize the feature vectors of each employee in a 2d/3d space using plotly

In [ ]:
import plotly.graph_objects as go

In [ ]:
from collections import Counter
#main topics for each cluster
cluster_titles = []

for i in range(TopicModelObject.k):
    k_sentences = np.array(sentences)[sentence_label_dict[i]]
    ngram_token_with_frequency = Counter(generate_n_gram(k_sentences, n=3, separator = '-' ,
                                        keep_stopwords= True, stem=False, keep_nouns_only=False))
    
    
    cluster_title = ', '.join([key for key, value in ngram_token_with_frequency.most_common(6)])
    
    cluster_titles.append(cluster_title)
cluster_titles

In [ ]:
np_sentences = np.array(sentences)

In [ ]:
fig = go.Figure()


for i, cl in enumerate(range(k)):
    print (i, cl)
    cluster_df = averaged_employee_features[averaged_employee_features.cluster==cl]
    #plot all the employees belonging to a cluster using a new trace. 
    features = np.stack(cluster_df['feature_vector_2d'].to_numpy())

    employee_titles = []
    # constructing hover text (all objectives belonging to that employee) 
    for emp in cluster_df.index.values:
        
        emp_indices = employee_objective_indices[emp]
        employee_title = '. '.join([objective for objective in np_sentences[emp_indices]])
    
        employee_titles.append(emp + ': ' + employee_title)
        
    fig.add_trace(go.Scattergl(
      x=features[:, 0],
      y=features[:, 1],
      mode='markers',
      showlegend=True,
      text = employee_titles,
      marker=dict(size=8, color=i),
      name="cluster_"+str(cl+1) +": " + cluster_titles[cl]
  ))



fig.update_layout(title='Distribution of employees over clusters in 2d space', legend=dict(
        x=0.6,
        y=1.4,
        traceorder='normal',
        font=dict(
            size=10)))
fig.show()


In [ ]:
fig = go.Figure()


for i, cl in enumerate(range(k)):
    cluster_df = averaged_employee_features[averaged_employee_features.cluster==cl]

    #plot all the employees belonging to a cluster using a new trace. 
    features = np.stack(cluster_df['feature_vector_2d'].to_numpy())
    #normalize the features between 0 and 1
    
    min_ = features.min(axis=0)
    max_ = features.max(axis=0)
    features_normed = (features - min_ )/ (max_ - min_) 
    
    employee_titles = []
    # constructing hover text (all objectives belonging to that employee) 
    for emp in cluster_df.index.values:
        
        emp_indices = employee_objective_indices[emp]
        employee_title = '. '.join([objective for objective in np_sentences[emp_indices]])
    
        employee_titles.append(emp + ': ' + employee_title)


    fig.add_trace(go.Scattergl(
      x=features_normed[:, 0],
      y=features_normed[:, 1],
      mode='markers',
      showlegend=True,
      text = employee_titles,
      marker=dict(size=8, color=i),
      name="cluster_"+str(cl+1) +": " + cluster_titles[cl]
    ))



fig.update_layout(title='Distribution of employees over clusters in 2d space - Normalized', legend=dict(
        x=0.6,
        y=1.4,
        traceorder='normal',
        font=dict(
            size=10)))
fig.show()

In [ ]:
fig = go.Figure()
for i , cl in enumerate(range(k)):
    
    cluster_df = averaged_employee_features[averaged_employee_features.cluster==cl]

    #plot all the employees belonging to a cluster using a new trace. 
    features = np.stack(cluster_df['feature_vector_3d'].to_numpy())
    
    employee_titles = []
    # constructing hover text (all objectives belonging to that employee) 
    for emp in cluster_df.index.values:
        
        emp_indices = employee_objective_indices[emp]
        employee_title = '. '.join([objective for objective in np_sentences[emp_indices]])
    
        employee_titles.append(emp + ': ' + employee_title)


    fig.add_trace(go.Scatter3d(
      x=features[:, 0],
      y=features[:, 1],
      z=features[:, 2],
      mode='markers',
      showlegend=True,
      text = employee_titles,
      marker=dict(size=3, color=i),
      name="cluster_"+str(cl+1) +": " + cluster_titles[cl]
    ))

fig.update_layout(title='Distribution of employees over clusters in 3d Space', legend=dict(
        x=0.6,
        y=1.4,
        traceorder='normal',
        font=dict(
            size=10)))
fig.show()

In [ ]:
fig = go.Figure()

for i, cl in enumerate(range(k)):
    cluster_df = averaged_employee_features[averaged_employee_features.cluster==cl]

    #plot all the employees belonging to a cluster using a new trace. 
    features = np.stack(cluster_df['feature_vector_3d'].to_numpy())
    
    print ("\n data \n ",features)
    min_ = features.min(axis=0)
    max_ = features.max(axis=0)
    print ("Row wise minimum: ", features.min(axis=0))
    print ("Row wise maximum: ", features.max(axis=0))
    features_normed = (features - min_ )/ (max_ - min_) 
    
    print ("Normalized\n", features_normed)
    
    employee_titles = []
    # constructing hover text (all objectives belonging to that employee) 
    for emp in cluster_df.index.values:
        
        emp_indices = employee_objective_indices[emp]
        employee_title = '. '.join([objective for objective in np_sentences[emp_indices]])
    
        employee_titles.append(emp + ': ' + employee_title)


    fig.add_trace(go.Scatter3d(
      x=features_normed[:, 0],
      y=features_normed[:, 1],
      z=features_normed[:, 2],
      mode='markers',
      showlegend=True,
      text = employee_titles,
      marker=dict(size=3, color=i),
      name="cluster_"+str(cl+1) +": " + cluster_titles[cl]
    ))

fig.update_layout(title='Distribution of employees over clusters in 3d Space - Normalized', legend=dict(
        x=0.6,
        y=1.4,
        traceorder='normal',
        font=dict(
            size=10)))
fig.show()

## 4.3 Visualization (Network)

In [35]:
from pyvis.network import Network
import networkx as nx

In [42]:
#create a dataframe to store the frequence per cluster for each employee. 
df_id_cluster = pd.DataFrame(0,
                  index=pd.Index(unique_ids, name='Employee id'),
                  columns=pd.Index([i for i in range(k)], name='Clusters'))
#There are multiple entries per employee so an employee can belong to multiple clusters
for i, _id in enumerate(ids):
    df_id_cluster[cluster_label_per_document[i]][_id] += 1
    
employee_cluster_frequency = df_id_cluster.values.tolist()
employee_cluster_frequency

[[12, 24], [7, 0], [60, 102]]

In [46]:
#create a dataframe to store the frequence per cluster for each employee. 
df_id_cluster = pd.DataFrame(0,
                  index=pd.Index(unique_ids, name='Employee id'),
                  columns=pd.Index([i for i in range(k)], name='Clusters'))
#There are multiple entries per employee so an employee can belong to multiple clusters
for i, _id in enumerate(ids):
    df_id_cluster[cluster_label_per_document[i]][_id] = 1
    
employee_cluster_frequency = df_id_cluster.to_numpy()


array([[1, 1],
       [1, 0],
       [1, 1]])

In [43]:
net = Network("500px", "500px")

In [44]:
# Create all cluster nodes

#cluster ids
cluster_node_ids = ["cluster_" + str(i+1) for i in range(k)]

#size of the cluster nodes
cluster_node_size = 1000

cluster_sizes = [cluster_node_size for _ in range(k)]
x = [40+i for i in range(k)]
y = [40+i for i in range(k)]
color = ["#dd4b39" for i in range(k)]

#main topics for each cluster
cluster_titles = []
for key, value in TopicModelObject.get_top_words_lda(top=10).items():
    cluster_title = ', '.join([item[0] for item in value])
    cluster_titles.append(cluster_title)

print(cluster_titles)


net.add_nodes(cluster_node_ids, value= cluster_sizes, 
              title=cluster_titles, 
              x=x, y=y, 
              label=cluster_node_ids, 
              color=color)


['time, team, regular, post, job, project, budget, area, continu, strategi', 'support, employe, biolog, initi, develop, appropri, talent, leadership, drive, busi']


In [45]:
# create all employee nodes

#employee ids
employee_node_ids = list(set(ids))

#size of the employee nodes
employee_node_size = 200

employee_node_sizes = [employee_node_size for _ in range(len(employee_node_ids))]
x = [10+i for i in range(len(employee_node_ids))]
y = [40+i for i in range(len(employee_node_ids))]
color = ["#162347" for i in range(len(employee_node_ids))]
 



# get all objective indices for each employee
employee_objective_indices = dict()
for employee in employee_node_ids:
    employee_indices = [i for i, e in enumerate(ids) if (ids[i] == employee) ]
    employee_objective_indices[employee] = employee_indices
    

#All objectives belong to an employee are concatenated
employee_titles = []
np_sentences = np.array(sentences)
for key, value in employee_objective_indices.items():
    employee_title = '. '.join([objective for objective in np_sentences[value]])
    
    employee_titles.append(employee_title)

subset= 3   

net.add_nodes(employee_node_ids[:subset], value=employee_node_sizes[:subset], 
              title=employee_titles[:subset], 
              x=x[:subset], y=y[:subset], 
              label=employee_node_ids[:subset], 
              color=color[:subset])

In [ ]:
# create all edges between employees and clusters
for emp,employee_cluster in enumerate(employee_cluster_frequency):
    for cl, cluster in enumerate(employee_cluster):
        
        # If the employee belongs to that cluster
        if cluster > 0:
            
            #print (employee_node_ids[emp],cluster_node_ids[cl])
            net.add_edge(employee_node_ids[emp],cluster_node_ids[cl],weight=float(cluster/num_docs))
            

In [ ]:
net.show_buttons(filter_=['physics'])

In [ ]:
# visualize the network
net.show("example.html")

### 4.4 pyLDAvis
This visualization only works on LDA results

In [ ]:
import pyLDAvis
from pyLDAvis import gensim as gs

In [ ]:
id2word = corpora.Dictionary(token_lists)
corpus = [id2word.doc2bow(text) for text in token_lists]

In [ ]:
pyLDAvis.enable_notebook()
vis = gs.prepare(TopicModelObject.ldamodel, corpus, id2word)
vis